## Topic 3: Why Qdrant Specifically

### Why Do We Need to Choose a Vector Database?

There are many vector databases available such as FAISS, Chroma, Qdrant, Pinecone, and Weaviate.

Each has different strengths.

Instead of asking **"Which is the best?"**, the correct question is:

**"Which one best satisfies our project requirements?"**

For this project, we needed a vector database that is:

- Easy to learn.
- Runs locally without cloud accounts.
- Supports metadata filtering.
- Can move from development to production easily.
- Open source.
- Production ready.

Qdrant satisfies all these requirements.

---

## Why We Chose Qdrant

### 1. Easy to Start

Qdrant can run completely inside Python using:

```python
QdrantClient(":memory:")
```

This means:

- No Docker.
- No server.
- No cloud account.
- No installation complexity.

Perfect for learning and development.

---

### 2. Easy to Move to Production

During learning we use:

```python
QdrantClient(":memory:")
```

Later, when persistence is needed, we only change one line:

```python
QdrantClient(url="http://localhost:6333")
```

Everything else remains exactly the same.

- Collections remain the same.
- Search code remains the same.
- Upsert code remains the same.
- Filtering remains the same.

This makes upgrading extremely easy.

---

### 3. Supports Metadata Filtering

Most production RAG applications do not search every document.

For example,

Suppose our knowledge base contains:

- Banking FAQs
- Insurance FAQs
- HR Policies
- Product Manuals

A banking question should search only banking documents.

Qdrant allows filtering during vector search.

Example:

Search only documents where

- Product = Fixed Deposit
- Language = English

This improves retrieval quality.

---

### 4. Persistent Storage

In-memory mode is useful for learning.

However, production applications cannot lose data after every restart.

Qdrant also supports persistent storage using Docker.

Once enabled, vectors remain available even after restarting the application.

---

### 5. Open Source

Qdrant is completely open source.

This provides:

- Full control.
- No vendor lock-in.
- Local deployment.
- Cloud deployment.
- Ability to inspect how the system works.

---

### 6. Production Ready

Qdrant is designed for production systems.

It provides:

- Persistent storage.
- Metadata filtering.
- Concurrent access.
- REST API.
- gRPC API.
- Docker deployment.
- Cloud deployment.

The same database can be used from development to production.

---

## Example

Suppose our customer support system stores one million FAQ chunks.

Each chunk contains:

- Embedding
- Original text
- Product
- Language
- Source PDF

When a customer asks about Fixed Deposits,

Qdrant searches only:

- Product = Fixed Deposit
- Language = English

instead of searching every document.

This produces faster and more relevant results.

---

## Advantages

- Very easy to start.
- Same client works in development and production.
- Rich metadata filtering.
- Open source.
- Production ready.
- Docker support.
- Cloud support.
- Excellent LangChain integration.

---

## Disadvantages

- More setup than Chroma.
- Docker is required for persistence.
- In-memory mode loses data after restart.
- Requires recreating collections when embedding dimensions change.

---

## Real-World Examples

### Learning

Students use:

```python
QdrantClient(":memory:")
```

because it requires almost zero setup.

---

### Startup

A startup runs Qdrant inside Docker on a single server.

No cloud cost.

Easy deployment.

---

### Enterprise

A banking chatbot stores millions of document embeddings inside Qdrant with metadata filters such as:

- Product
- Department
- Language
- Country

Only relevant documents are searched.

---

## Real-World Issues, Edge Cases & Debugging

### In-Memory Mode

Problem

All vectors disappear after restarting Python.

Reason

`:memory:` does not provide persistence.

Solution

Use Docker deployment.

---

### Collection Already Exists

Problem

Creating the same collection twice throws an exception.

Solution

Check whether the collection already exists before creating it.

---

### Vector Dimension Mismatch

Problem

Documents were embedded using one embedding model, while queries use another.

Reason

Vector dimensions are different.

Solution

Recreate the collection using the correct embedding model.

---

### Missing Docker Volume

Problem

Vectors disappear after restarting the Docker container.

Reason

Persistent storage was not mounted.

Solution

Always use Docker volumes for production deployments.

---

## Common Mistakes & Production Failures

- Using `:memory:` in production.
- Forgetting Docker volume mounting.
- Changing embedding models without rebuilding collections.
- Ignoring metadata filtering.
- Assuming Docker and in-memory mode behave differently.
- Not checking vector dimensions before ingestion.

---

## Lead-Level Interview Questions

### Q1. Why did you choose Qdrant instead of Chroma?

**Answer**

Chroma is excellent for prototyping, but Qdrant provides a smoother transition to production. It supports persistence, better metadata filtering, concurrent access, Docker deployment, and cloud deployment while keeping almost the same client code.

---

### Q2. Why is `QdrantClient(":memory:")` useful?

**Answer**

It allows developers to learn and build applications without installing Docker or managing servers. Once persistence is required, only the client configuration changes while the application logic remains unchanged.

---

### Q3. What happens if you change the embedding model?

**Answer**

Changing the embedding model usually changes the vector dimensions.

Existing collections cannot store vectors of different dimensions.

The collection must be recreated and all documents must be re-embedded.

---

### Q4. Why is metadata filtering important?

**Answer**

Searching every document wastes computation and reduces retrieval quality.

Metadata filtering limits the search space to relevant documents, improving both speed and accuracy.

---

### Q5. Why is Qdrant a good choice for production?

**Answer**

Because it provides persistent storage, metadata filtering, concurrent access, REST and gRPC APIs, Docker deployment, cloud deployment, and an easy migration path from development to production without changing application logic.

---

## Key Takeaways

- Qdrant was chosen because it matches this project's requirements.
- `:memory:` mode is ideal for learning and development.
- Docker provides persistence for production.
- Metadata filtering improves retrieval quality.
- The same application code works in development, testing, and production.
- Only the client configuration changes when moving between deployment modes.

In [1]:
"""
qdrant_setup.py
-----------------
Qdrant setup and connection verification.

Uses :memory: mode -- no Docker required, no network download,
starts instantly inside the Python process. Full Qdrant behavior
(collections, points, payloads, filtering, search) works identically
in :memory: mode and Docker mode.

The ONLY difference between modes is one line in get_client():
  Learning / no Docker : QdrantClient(":memory:")
  Local Docker         : QdrantClient(url="http://localhost:6333")
  Qdrant Cloud         : QdrantClient(url="https://...", api_key="...")

When you want persistence:
  docker run -p 6333:6333 -v <your_path>/qdrant_storage:/qdrant/storage qdrant/qdrant
  Then swap get_client() to QdrantClient(url="http://localhost:6333").
  Nothing else in this file or any other file needs to change.

Install: pip install qdrant-client
"""

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

COLLECTION_NAME = "fd_knowledge_base"
VECTOR_SIZE = 384        # paraphrase-multilingual-MiniLM-L12-v2 output dimension


def get_client() -> QdrantClient:
    """Returns a Qdrant client.

    Currently uses :memory: mode -- no Docker needed, zero setup,
    data is lost when the Python process restarts.

    To switch to persistent local Docker, replace with:
        return QdrantClient(url="http://localhost:6333")

    To switch to Qdrant Cloud, replace with:
        return QdrantClient(url="https://your-cluster.qdrant.io", api_key="YOUR_KEY")
    """
    return QdrantClient(":memory:")


def create_collection(client: QdrantClient, recreate: bool = False) -> None:
    """Creates the collection if it does not already exist.

    recreate=True drops and rebuilds the collection -- use this when
    changing embedding models or chunk schemas. Never use on a live
    collection with real data.
    """
    # check what collections already exist so we do not crash on a duplicate name
    existing = [c.name for c in client.get_collections().collections]

    if COLLECTION_NAME in existing:
        if recreate:
            # deliberate teardown -- only when we know we want a clean slate
            client.delete_collection(COLLECTION_NAME)
            print(f"Deleted existing collection '{COLLECTION_NAME}'")
        else:
            print(f"Collection '{COLLECTION_NAME}' already exists -- skipping creation")
            return

    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=VECTOR_SIZE,           # every vector in this collection must be 384 floats
            distance=Distance.COSINE,   # cosine similarity -- right choice for normalized vectors
        ),
    )
    print(f"Created collection '{COLLECTION_NAME}' (vector_size={VECTOR_SIZE})")


def verify_connection(client: QdrantClient) -> None:
    """Quick sanity check -- prints collection info if everything is working."""
    info = client.get_collection(COLLECTION_NAME)
    print(f"Collection   : {COLLECTION_NAME}")
    print(f"Vector size  : {info.config.params.vectors.size}")
    print(f"Points count : {info.points_count}")
    print(f"Status       : {info.status}")


# ----------------------------------------------------------------------
# Run setup
# ----------------------------------------------------------------------

# get a client -- :memory: mode, no Docker needed
client = get_client()

# create the collection
create_collection(client, recreate=False)

# confirm everything is working
verify_connection(client)

Created collection 'fd_knowledge_base' (vector_size=384)
Collection   : fd_knowledge_base
Vector size  : 384
Points count : 0
Status       : green
